# Sleep Duration Prediction – Model Selection

**Kaggle username:** Ran Tao 2026

## Introduction

The aim of this exercise is to predict `Sleep_Duration` from demographic, lifestyle and health-related descriptors.

To satisfy the model selection requirement, I use Lasso regression as a shrinkage method and compare its predictive performance with a standard linear regression model and a Random Forest model. Model performance is evaluated using mean squared error (MSE).

The final predictions were submitted to the Kaggle competition.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


## Load the data

The training data contain the response variable `Sleep_Duration`. The test data contain the observations for which sleep duration should be predicted.


In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)

train.head()


## Data preparation

The `ID` variable is an identifier and is excluded from the predictors.

The dataset contains both numerical and categorical predictors. Numerical variables are standardized for the linear and shrinkage models. Missing values are handled during preprocessing, and categorical variables are transformed using one-hot encoding.


In [ ]:
print(train.info())

print("\nMissing values:")
print(train.isnull().sum())


In [ ]:
X = train.drop(columns=["Sleep_Duration", "ID"])
y = train["Sleep_Duration"]

X_test = test.drop(columns=["ID"])

categorical_cols = X.select_dtypes(include="object").columns.tolist()
numerical_cols = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical variables:", categorical_cols)
print("Numerical variables:", numerical_cols)


## Training and validation data

The available training observations are randomly divided into a training set (80%) and a validation set (20%). A fixed random seed is used so that the analysis is reproducible. All models are evaluated on the same validation observations.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training observations:", len(X_train))
print("Validation observations:", len(X_val))


In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])


## Model selection

Three predictive approaches are considered:

1. Ordinary linear regression as a baseline.
2. Lasso regression as a shrinkage method.
3. Random Forest regression as a flexible nonlinear predictive model.

Lasso uses an L1 penalty,

\[
\sum_i (y_i-\hat y_i)^2+\lambda\sum_j|\beta_j|,
\]

which shrinks regression coefficients and can set some coefficients exactly to zero. Thus, Lasso performs variable selection. The penalty parameter is selected using cross-validation.

The models are compared using validation mean squared error,

\[
MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2.
\]

A smaller MSE indicates better predictive performance.


### 1. Linear regression

Ordinary linear regression is used as a baseline against which the more flexible approaches can be compared.


In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_val)

linear_mse = mean_squared_error(y_val, linear_pred)

print("Linear Regression validation MSE:", linear_mse)


### 2. Lasso regression

Lasso is the shrinkage-based model selection method used in this exercise. The penalty parameter `alpha` is selected by 5-fold cross-validation using only the training portion of the data.


In [ ]:
lasso_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso(max_iter=100000))
])

lasso_parameters = {
    "model__alpha": np.logspace(-4, 0, 50)
}

lasso_search = GridSearchCV(
    lasso_model,
    lasso_parameters,
    cv=5,
    scoring="neg_mean_squared_error"
)

lasso_search.fit(X_train, y_train)

lasso_pred = lasso_search.best_estimator_.predict(X_val)
lasso_mse = mean_squared_error(y_val, lasso_pred)

print("Best Lasso alpha:", lasso_search.best_params_["model__alpha"])
print("Lasso validation MSE:", lasso_mse)


### 3. Random Forest

Random Forest is also evaluated because it can represent nonlinear relationships and interactions between predictors without specifying them explicitly.


In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_val)

rf_mse = mean_squared_error(y_val, rf_pred)

print("Random Forest validation MSE:", rf_mse)


## Comparison of predictive performance

The models are compared on exactly the same validation observations. The model with the lowest validation MSE has the best predictive performance on this split.


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Lasso Regression",
        "Random Forest"
    ],
    "Validation MSE": [
        linear_mse,
        lasso_mse,
        rf_mse
    ]
}).sort_values("Validation MSE").reset_index(drop=True)

results


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(results["Model"], results["Validation MSE"])
plt.ylabel("Validation MSE")
plt.title("Comparison of Predictive Models")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## Model choice

Lasso regression was included as a shrinkage-based model selection method, and its penalty parameter was selected using cross-validation.

The predictive models were then compared on the same validation dataset using MSE. The model with the lowest validation MSE is preferred because it gives the best out-of-sample predictive performance among the models considered.

In the analysis of this dataset, Random Forest gives strong validation performance and was the model used for the Kaggle prediction. Its flexibility allows it to capture nonlinear relationships and interactions that are not represented by ordinary linear regression.

A limitation of this comparison is the relatively small dataset. Consequently, the exact validation MSE can depend on the particular train-validation split.


## Final Random Forest model

After model evaluation, the Random Forest model is fitted to all available training observations. This allows all available labelled data to contribute to the final predictive model.


In [ ]:
rf_model.fit(X, y)
predictions = rf_model.predict(X_test)

print("Number of predictions:", len(predictions))
predictions[:10]


## Prediction file

The following code creates a prediction file from the fitted Random Forest model. The Kaggle competition has already ended, so this file does not replace the historical submission. The original `submission.csv` supplied with this notebook is the file that was actually submitted during the competition.


In [ ]:
submission_reproduced = pd.DataFrame({
    "ID": test["ID"],
    "Sleep_Duration": predictions
})

submission_reproduced.to_csv(
    "submission_reproduced.csv",
    index=False
)

submission_reproduced.head()


## Historical Kaggle submission

The original `submission.csv` is retained as the historical Kaggle submission. The following cell checks its dimensions and displays the first observations.


In [ ]:
kaggle_submission = pd.read_csv("submission.csv")

print("Historical submission shape:", kaggle_submission.shape)
kaggle_submission.head()


## Conclusion

In this exercise, predictive models for sleep duration were developed and evaluated using mean squared error.

Lasso regression was used as a shrinkage method for model selection, with its penalty parameter selected by cross-validation. Its predictive performance was compared with ordinary linear regression and Random Forest regression on a common validation set.

Random Forest showed strong validation performance and was used for the Kaggle prediction. The historical Kaggle submission contains predicted sleep durations for all observations in the test dataset.

The analysis demonstrates the value of model selection and out-of-sample evaluation: alternative model specifications should be compared using predictive performance rather than only their fit to the training data.
